### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "predictors_vs_threshold" / "weighted" / "XGBoost_thres0.5_2026-05-11" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.94,,,,,
1,0,cf_1,,,,,,,18.2281,,,0.0746,1,False,0.3288,0.2555
2,0,cf_2,,,,,,,17.7157,5,,0.2143,2,False,0.3288,0.2092
3,0,cf_3,,,,7,,,18.9745,7,,0.2512,3,False,0.3288,0.4485
4,0,cf_4,,,5,,,,18.9619,2,,0.1215,3,False,0.3288,0.2091
5,0,cf_5,2,,,,,,18.9619,3,,0.181,3,False,0.3288,0.2337
6,0,cf_6,2,,,,,,18.9619,6,,0.2346,3,False,0.3288,0.3587
7,0,cf_7,2,,,5,,,18.9619,,,0.1691,3,False,0.3288,0.3939
8,0,cf_8,3,,,,,,18.9619,3,,0.1185,3,False,0.3288,0.2244
9,0,cf_9,,1,,,,,18.9619,3,,0.181,3,False,0.3288,0.1953


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.94,,,,,
30,0,cf_1,,,,,,,18.2281,,,0.0746,1,False,0.3288,0.2555
1,1,xin,3,4,1,2,3,0,22.3757,0,0.95,,,,,
9,1,cf_2,,,,,1,,22.3757,1,,0.1475,3,True,0.6788,0.3031
10,1,cf_5,,,,3,1,,22.3757,7,,0.3796,4,True,0.6788,0.3355
11,1,cf_6,,3,,,1,,22.3757,4,,0.2427,4,True,0.6788,0.2163
12,1,cf_8,2,,,,2,,22.3757,3,,0.1832,4,True,0.6788,0.2871
13,1,cf_9,,,,3,2,,22.3757,2,,0.2278,4,True,0.6788,0.332
14,1,cf_10,1,1,,,1,,22.3757,3,,0.4332,5,True,0.6788,0.1535
2,2,xin,5,3,1,1,4,0,22.694,7,0.99,,,,,


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.94,,,,,
9,0,cf_1,,,,,,,18.2281,,,0.0746,1,False,0.3288,0.2555
1,1,xin,3,4,1,2,3,0,22.3757,0,0.95,,,,,
10,1,cf_2,,,,,1,,22.3757,1,,0.1475,3,True,0.6788,0.3031
2,2,xin,5,3,1,1,4,0,22.694,7,0.99,,,,,
11,2,cf_4,1,,,,1,,22.6757,,,0.2513,3,True,0.7579,0.3559
3,3,xin,3,3,6,6,2,0,24.3418,1,1.02,,,,,
12,3,cf_1,,,,,,,24.3375,3,,0.0627,2,False,0.5708,0.4794
4,4,xin,5,4,2,7,1,0,21.2585,3,1.04,,,,,
13,4,cf_1,,,,,,,21.2585,,,0.1118,1,False,0.1406,0.1305


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.2555
Valid: False
Changes:
  bmi: 18.9866 → 18.2281


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_2 ---
Predicted risk: 0.3031
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 1

--- cf_5 ---
Predicted risk: 0.3355
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 1
  dosprt: 0 → 7

--- cf_6 ---
Predicted risk: 0.2163
Valid: True
Changes:
  eatveg: 4 → 3
  slprl: 3 → 1
  dosprt: 0 → 4

--- cf_8 ---
Predicted risk: 0.2871
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 2
  dosprt: 0 → 3

--- cf_9 ---
Predicted risk: 0.332
Valid: True
Changes:
  alcfreq: